# 01. BPv7 기초: DTN, 결정적 CBOR 부분집합, 수명 판정

작성·검증 기준일: **2026-09-11**  
원문: [RFC 9171 — Bundle Protocol Version 7](https://www.rfc-editor.org/rfc/rfc9171.html)

## 학습 목표

1. DTN의 `store-carry-forward`와 BPA/CLA/AA의 역할을 설명한다.
2. Python 표준 라이브러리만으로 **교육용 CBOR 부분집합**을 결정적으로 인코딩·디코딩한다.
3. 생성 시각, Bundle Age 블록, lifetime으로 만료 여부를 판정한다.

> **범위 경계:** 아래 코드는 RFC 9171 전체 wire format 구현이 아니다. CBOR 코드는 unsigned integer, byte/text string, definite-length array만 지원하는 엄격한 **toy subset**이다. BPv7의 indefinite-length outer array, EID tag/표현 전체, CRC/BPSec, 모든 플래그를 직렬화하지 않으므로 실제 노드와 상호운용하지 않는다.

관련 절: RFC 9171 §1, §3.1–3.2, §4.1, §4.2.6–4.2.7, §4.4.2, §5.5.

실행: Jupyter에서 **Run All**을 누른다. 외부 패키지와 네트워크는 필요 없다.

## 1. 지연 허용 네트워킹을 사건으로 생각하기

인터넷의 즉시 연결을 가정하기 어려운 우주·재난·센서 환경에서는 중간 노드가 bundle을 저장했다가 다음 contact 때 전달한다. RFC 9171의 bundle node는 개념적으로 다음 요소를 가진다.

- **AA (Application Agent)**: 애플리케이션 데이터 단위(ADU)를 만들고 소비한다.
- **BPA (Bundle Protocol Agent)**: bundle 생성·수신·전달·삭제 절차를 수행한다.
- **CLA (Convergence-Layer Adapter)**: TCPCL 등 하위 전송 수단으로 이웃 노드와 bundle을 주고받는다.

다음 작은 시뮬레이션은 라우팅 알고리즘이 아니라, 연결이 없는 동안 데이터를 보관한다는 개념만 보여 준다.

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Contact:
    sender: str
    receiver: str
    opens_at: int
    closes_at: int

def store_carry_forward(path: list[Contact], created_at: int) -> list[str]:
    """예약 contact를 따라 기다린 시간을 기록하는 교육용 모델이다."""
    now = created_at
    events: list[str] = []
    for contact in path:
        if contact.closes_at < now:
            raise ValueError(f"이미 닫힌 contact: {contact}")
        if now < contact.opens_at:
            events.append(f"t={now}..{contact.opens_at}: {contact.sender}가 저장")
            now = contact.opens_at
        events.append(f"t={now}: {contact.sender} -> {contact.receiver} 전달")
        now += 1  # 이 toy에서는 전송에 1 시간 단위가 든다고 가정한다.
    return events

contacts = [
    Contact("earth", "relay", 4, 8),
    Contact("relay", "mars", 30, 35),
]
events = store_carry_forward(contacts, created_at=0)
print(*events, sep="\n")

t=0..4: earth가 저장
t=4: earth -> relay 전달
t=5..30: relay가 저장
t=30: relay -> mars 전달


## 2. 결정적 CBOR의 최소 부분집합

RFC 9171 §4.1은 bundle 필드가 RFC 8949의 core deterministic encoding 요구를 따르도록 한다. 같은 값이 같은 바이트열이 되어야 블록 무결성 계산을 재현할 수 있기 때문이다. 다음 encoder는 정수에 가능한 가장 짧은 additional-information 표현을 사용한다.

지원 범위는 `unsigned int`, `bytes`, `str`, `list/tuple`뿐이다. map, tag, 음수, 부동소수점, indefinite-length 항목은 의도적으로 거부한다.

In [2]:
def _encode_head(major: int, value: int) -> bytes:
    if not 0 <= major <= 7 or value < 0:
        raise ValueError("major와 value는 허용 범위여야 합니다")
    if value < 24:
        return bytes([(major << 5) | value])
    if value <= 0xFF:
        return bytes([(major << 5) | 24, value])
    if value <= 0xFFFF:
        return bytes([(major << 5) | 25]) + value.to_bytes(2, "big")
    if value <= 0xFFFFFFFF:
        return bytes([(major << 5) | 26]) + value.to_bytes(4, "big")
    if value <= 0xFFFFFFFFFFFFFFFF:
        return bytes([(major << 5) | 27]) + value.to_bytes(8, "big")
    raise OverflowError("이 toy encoder는 uint64까지만 지원합니다")

def cbor_encode(value) -> bytes:
    """RFC 8949 결정적 표현을 따르는 작은 CBOR 부분집합 encoder."""
    if type(value) is int:  # bool은 int의 하위 타입이므로 정확한 타입을 검사한다.
        return _encode_head(0, value)
    if isinstance(value, bytes):
        return _encode_head(2, len(value)) + value
    if isinstance(value, str):
        encoded = value.encode("utf-8")
        return _encode_head(3, len(encoded)) + encoded
    if isinstance(value, (list, tuple)):
        return _encode_head(4, len(value)) + b"".join(cbor_encode(v) for v in value)
    raise TypeError(f"지원하지 않는 CBOR toy 타입: {type(value).__name__}")

In [3]:
class CborDecodeError(ValueError):
    pass

def cbor_decode(data: bytes, *, max_depth: int = 16, max_items: int = 1024,
                max_blob_bytes: int = 1 << 20):
    """비최소 정수 표현과 trailing bytes를 거부하는 bounded toy decoder."""
    if not isinstance(data, bytes):
        raise TypeError("data는 bytes여야 합니다")
    view = memoryview(data)
    budget = [max_items]

    def read_uint(ai: int, pos: int) -> tuple[int, int]:
        widths = {24: 1, 25: 2, 26: 4, 27: 8}
        if ai < 24:
            return ai, pos
        if ai not in widths:
            raise CborDecodeError("indefinite/reserved additional info는 이 toy에서 금지")
        width = widths[ai]
        if pos + width > len(view):
            raise CborDecodeError("잘린 정수")
        value = int.from_bytes(view[pos:pos + width], "big")
        minima = {24: 24, 25: 256, 26: 65536, 27: 1 << 32}
        if value < minima[ai]:
            raise CborDecodeError("결정적 CBOR가 아닌 비최소 정수 표현")
        return value, pos + width

    def parse(pos: int, depth: int):
        if depth > max_depth:
            raise CborDecodeError("최대 중첩 깊이 초과")
        if pos >= len(view):
            raise CborDecodeError("입력이 중간에서 끝남")
        budget[0] -= 1
        if budget[0] < 0:
            raise CborDecodeError("최대 항목 수 초과")
        initial = view[pos]
        pos += 1
        major, ai = initial >> 5, initial & 0x1F
        value, pos = read_uint(ai, pos)
        if major == 0:
            return value, pos
        if major in (2, 3):
            if value > max_blob_bytes or pos + value > len(view):
                raise CborDecodeError("문자열 길이가 제한을 넘거나 입력이 잘림")
            raw = bytes(view[pos:pos + value])
            if major == 2:
                return raw, pos + value
            try:
                return raw.decode("utf-8"), pos + value
            except UnicodeDecodeError as exc:
                raise CborDecodeError("잘못된 UTF-8") from exc
        if major == 4:
            if value > max_items:
                raise CborDecodeError("배열 길이 제한 초과")
            result = []
            for _ in range(value):
                item, pos = parse(pos, depth + 1)
                result.append(item)
            return result, pos
        raise CborDecodeError(f"major type {major}는 이 toy에서 지원하지 않음")

    result, end = parse(0, 0)
    if end != len(view):
        raise CborDecodeError("하나의 값 뒤에 trailing bytes가 있음")
    return result

In [4]:
examples = [0, 23, 24, 255, 256, b"BPv7", "dtn://node/app", [7, 0, [1, "node"]]]
for value in examples:
    wire = cbor_encode(value)
    decoded = cbor_decode(wire)
    expected = list(value) if isinstance(value, tuple) else value
    assert decoded == expected
    print(f"{value!r:30} -> {wire.hex()}")

# 24를 2바이트 argument로 표현한 0x19 0018은 값은 같아도 비결정적이므로 거부한다.
try:
    cbor_decode(bytes.fromhex("190018"))
except CborDecodeError as exc:
    print("비최소 표현 거부:", exc)

0                              -> 00
23                             -> 17
24                             -> 1818
255                            -> 18ff
256                            -> 190100
b'BPv7'                        -> 4442507637
'dtn://node/app'               -> 6e64746e3a2f2f6e6f64652f617070
[7, 0, [1, 'node']]            -> 8307008201646e6f6465
비최소 표현 거부: 결정적 CBOR가 아닌 비최소 정수 표현


## 3. Bundle age와 lifetime

RFC 9171 §5.5의 경계는 `age exceeds lifetime`, 즉 **age > lifetime**이다. 따라서 정확히 같은 값이면 아직 만료로 판정하지 않는다.

- 생성 시각이 0이 아니고 로컬 시계가 정확하면 `현재 DTN 시각 - 생성 DTN 시각`을 사용할 수 있다.
- 그렇지 않으면 Bundle Age extension block 값이 필요하다.
- 생성 시각이 0이면 §4.4.2에 따라 Bundle Age 블록이 정확히 하나 있어야 한다.
- 구현은 DoS 방어를 위해 primary block을 바꾸지 않고 더 짧은 local effective lifetime을 적용할 수 있다.

In [5]:
@dataclass(frozen=True)
class AgeContext:
    creation_time_ms: int
    lifetime_ms: int
    clock_accurate: bool
    bundle_age_blocks_ms: tuple[int, ...] = ()

def bundle_age_ms(context: AgeContext, current_dtn_time_ms: int) -> int:
    values = (context.creation_time_ms, context.lifetime_ms, current_dtn_time_ms,
              *context.bundle_age_blocks_ms)
    if any(type(value) is not int or value < 0 for value in values):
        raise ValueError("시각·수명·age는 음이 아닌 정수(ms)여야 합니다")
    if len(context.bundle_age_blocks_ms) > 1:
        raise ValueError("Bundle Age 블록은 둘 이상 올 수 없습니다 (§4.4.2)")
    if context.creation_time_ms == 0:
        if len(context.bundle_age_blocks_ms) != 1:
            raise ValueError("creation time=0이면 Bundle Age 블록이 정확히 하나 필요합니다")
        return context.bundle_age_blocks_ms[0]
    if context.clock_accurate:
        if current_dtn_time_ms < context.creation_time_ms:
            raise ValueError("정확하다고 선언한 시계가 생성 시각보다 과거입니다")
        return current_dtn_time_ms - context.creation_time_ms
    if len(context.bundle_age_blocks_ms) != 1:
        raise ValueError("부정확한 시계에서는 Bundle Age 블록이 필요합니다")
    return context.bundle_age_blocks_ms[0]

def expiration(context: AgeContext, now_ms: int, local_override_ms: int | None = None):
    effective = context.lifetime_ms
    reason = "Lifetime expired"
    if local_override_ms is not None:
        if type(local_override_ms) is not int or local_override_ms < 0:
            raise ValueError("override는 음이 아닌 정수여야 합니다")
        if local_override_ms < effective:
            effective = local_override_ms
            reason = "Traffic pared"
    age = bundle_age_ms(context, now_ms)
    return {"age_ms": age, "effective_lifetime_ms": effective,
            "expired": age > effective, "reason_if_expired": reason}

known_clock = AgeContext(creation_time_ms=1_000, lifetime_ms=5_000, clock_accurate=True)
unknown_clock = AgeContext(creation_time_ms=0, lifetime_ms=5_000,
                           clock_accurate=False, bundle_age_blocks_ms=(5_001,))
print("경계(age==lifetime):", expiration(known_clock, 6_000))
print("age block 기반 만료:", expiration(unknown_clock, 999_999))
print("로컬 수명 override:", expiration(known_clock, 3_500, local_override_ms=2_000))

경계(age==lifetime): {'age_ms': 5000, 'effective_lifetime_ms': 5000, 'expired': False, 'reason_if_expired': 'Lifetime expired'}
age block 기반 만료: {'age_ms': 5001, 'effective_lifetime_ms': 5000, 'expired': True, 'reason_if_expired': 'Lifetime expired'}
로컬 수명 override: {'age_ms': 2500, 'effective_lifetime_ms': 2000, 'expired': True, 'reason_if_expired': 'Traffic pared'}


## 4. 자동 검수와 예상 결과

아래 셀이 `기초 실습 검수 통과`를 출력하면 다음을 확인한 것이다.

- 결정적 CBOR 부분집합이 알려진 짧은 벡터와 round-trip을 통과한다.
- 비최소 표현과 trailing bytes가 거부된다.
- lifetime 경계가 `>`로 처리된다.
- 시계가 없을 때 age block 누락이 조용히 0으로 처리되지 않는다.

In [6]:
assert cbor_encode(23) == bytes.fromhex("17")
assert cbor_encode(24) == bytes.fromhex("1818")
assert cbor_decode(cbor_encode([1, b"x", "한글"])) == [1, b"x", "한글"]
for invalid in (bytes.fromhex("1817"), bytes.fromhex("00ff")):
    try:
        cbor_decode(invalid)
    except CborDecodeError:
        pass
    else:
        raise AssertionError(f"거부되어야 할 입력: {invalid.hex()}")
assert expiration(known_clock, 6_000)["expired"] is False
assert expiration(known_clock, 6_001)["expired"] is True
try:
    bundle_age_ms(AgeContext(0, 10, False), 100)
except ValueError:
    pass
else:
    raise AssertionError("필수 Bundle Age 블록 누락을 거부해야 합니다")
print("기초 실습 검수 통과")

기초 실습 검수 통과


## 연습 문제

1. `cbor_encode`에 음수 정수(major type 1)를 추가하되 최소 길이 검사를 유지하라.
2. `max_depth=2`인 decoder에 3중 배열을 넣고 거부되는지 확인하라.
3. contact가 열려 있는 시간이 전송 시간보다 짧을 때 다음 기회를 기다리도록 시뮬레이터를 확장하라.
4. 로컬 lifetime override가 primary block의 원래 lifetime을 변경하지 않는 이유를 설명하라.

다음 노트북에서는 primary/canonical block 모델, 정확한 CRC 체크 벡터, fragmentation/reassembly를 다룬다.